# A generic branch-and-bound algorithm for l0-regularized problems.

*C. Elvira, T. Guyard, and C. Herzet. Submitted in 2025.*

#### Notebook to reproduce experiments of Section 6.1

---

This experiment compares the performance of different methods to fit a regularization path for on L0-regularized problems with real-worlds data. To reproduce it, let first import all the necessary packages and routines.

In [ ]:
from el0ps.datafit import Leastsquares, Logistic, Squaredhinge
from el0ps.penalty import BigmL1L2norm, BigmL2norm
from el0ps.path import Path
from el0ps.utils import compute_lmbd_max
from l0exp.dataset import get_dataset_realworld
from l0exp.solver import get_solver, can_handle_instance

Now, let select the dataset to use. Each dataset contains a feature matrix `A` and a target vector `y`. Possible choices are:
- `dexter`: Text classification problem from the NIPS 2003 feature selection challenge.
- `dorothea`: Drug discovery problem from the NIPS 2003 feature selection challenge.
- `drug`: Drug discovery problem from "DeepCDR: a hybrid graph convolutional network for predicting cancer drug response" by Quiao Liu et al. (2020).
- `gisette`: Handwritten digit recognition problem from the NIPS 2003 feature selection challenge.
- `kits`: Kidney tumor classification dataset from the KiTS19 challenge.
- `madelon`: Synthetic dataset from the NIPS 2003 feature selection challenge.
By default, datasets are normalized.

In [ ]:
dataset = "kits"
A, y, x = get_dataset_realworld(name=dataset, normalize=True)
print(f"Dataset: {dataset}")
print(f"A shape: {A.shape}")
print(f"y shape: {y.shape}")

We can next define the problem data-fidelity and penalty functions, as well as the L0-norm weight involved in the L0-regularized problem. In our experiments, the following values have been used:

| Dataset   | Data-fidelity            | Penalty                                                                  | L0-norm weight `lmbd`   |
|-----------|--------------------------|--------------------------------------------------------------------------|-------------------------|
| dexter    | `Squaredhinge(y)`        | `BigmL1L2norm(M=1.4109822944408792, alpha=0.3, beta=0.03)`               | `4.151753202099959`     |
| dexter    | `Squaredhinge(y)`        | `BigmL2norm(M=1.4109822944408792, beta=3.0)`                             | `0.6396638097343027`    |
| dorothea  | `Squaredhinge(y)`        | `BigmL1L2norm(M=0.7981452920212397, alpha=0.8, beta=0.8)`                | `6.588191587399787`     |
| dorothea  | `Squaredhinge(y)`        | `BigmL2norm(M=0.7981452920212397, beta=0.8)`                             | `6.817662423893168`     |
| drug      | `Leastsquares(y)`        | `BigmL1L2norm(M=9.06233865614505, alpha=0.822, beta=8.22)`               | `0.2106722787843218`    |
| drug      | `Leastsquares(y)`        | `BigmL2norm(M=4.531169328072525, beta=8.22)`                             | `0.00315234`            |
| gisette   | `Logistic(y)`            | `BigmL1L2norm(M=4912.651076567635, alpha=0.6, beta=0.6)`                 | `18.885741683133276`    |
| gisette   | `Logistic(y)`            | `BigmL2norm(M=491.2651076567635, beta=0.6)`                              | `20.114872094098644`    |
| kits      | `Leastsquares(y)`        | `BigmL1L2norm(M=0.5759839750528222, alpha=0.1, beta=1.0)`                | `0.04728410277698789`   |
| kits      | `Leastsquares(y)`        | `BigmL2norm(M=0.5759839750528222, beta=1.0)`                             | `0.08047734686862017`   |
| madelon   | `Logistic(y)`            | `BigmL1L2norm(M=6.85122771618553, alpha=0.2, beta=0.2)`                  | `0.3255550588532803`    |
| madelon   | `Logistic(y)`            | `BigmL2norm(M=13.70245543237106, beta=0.2)`                              | `0.7585954127924406`    |

In [ ]:
datafit = Leastsquares(y)
penalty = BigmL1L2norm(M=0.5759839750528222, alpha=0.1, beta=1.0)
lmbd    = 0.04728410277698789
print(f"datafit   : {datafit}")
print(f"penalty   : {penalty}")
print(f"lambda    : {lmbd}")
print(f"lambda_max: {compute_lmbd_max(datafit, penalty, A)}")

Finally, we can define the different methods to compare and their parameters. Here is the setup used in our experiments. Solvers that cannot handle the considered instance are automatically skipped.

In [ ]:
# Solvers to use (comment out those you don't want to run or that are not installed)
solver_types = [
    "el0ps",
    "l0bnb",
    "mimosa",
    "gurobi",
    "mosek",
    "oa"
]

solver_args = {
    "time_limit"    : 3600,   # time limit in seconds for each value of lambda in the path
    "relative_gap"  : 1.e-8,  # relative optimality gap on the objective value
    "verbose"       : False,  # verbosity toggle for solvers
}

path_args = {
    "lmbd_max"          : 1.0,      # maximum value of lmbd / lmbd_max in the path
    "lmbd_min"          : 0.01,     # minimum value of lmbd / lmbd_max in the path
    "lmbd_num"          : 20,       # number of values of lmbd in the path
    "lmbd_normalized"   : True,     # normalize lmbd w.r.t lmbd_max in the path
}

for solver_type in solver_types:

    try:    
        solver = get_solver(solver_type, solver_args)
        if can_handle_instance(solver, datafit, penalty):
            print(f"Running {solver_type}...")
            path = Path(**path_args)
            path.fit(solver, datafit, penalty, A)
        else:
            print(f"Skipping {solver_type}: cannot handle this instance")
    except Exception as e:
        print(f"Solver {solver_type} failed with error: {e}")
    print()